<a href="https://colab.research.google.com/github/12halima/Transport_Recommander/blob/main/Process_GTFS-OSM/Network_Base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GTFS_Network_Creation") \
    .getOrCreate()

print("Spark version:", spark.version)


Spark version: 4.0.1


In [6]:
# ===============================
# 2. Définition des chemins GTFS
# ===============================

# Dossier principal dans Google Drive
BASE_PATH = "/content/drive/MyDrive/GTFS_FINAL"

# Dossier contenant les GTFS nettoyés par ville
CLEAN_PATH = f"{BASE_PATH}/GTFS_CLEAN"

# Fichier stop_times global (33+ millions de lignes)
STOP_TIMES_PATH = f"{BASE_PATH}/stop_times_final.csv"

print("BASE_PATH :", BASE_PATH)
print("CLEAN_PATH :", CLEAN_PATH)
print("STOP_TIMES_PATH :", STOP_TIMES_PATH)


BASE_PATH : /content/drive/MyDrive/GTFS_FINAL
CLEAN_PATH : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN
STOP_TIMES_PATH : /content/drive/MyDrive/GTFS_FINAL/stop_times_final.csv


In [4]:
import os

print("GTFS_FINAL existe :", os.path.exists(BASE_PATH))
print("GTFS_CLEAN existe :", os.path.exists(CLEAN_PATH))
print("stop_times_final.csv existe :", os.path.exists(STOP_TIMES_PATH))



GTFS_FINAL existe : True
GTFS_CLEAN existe : True
stop_times_final.csv existe : True


In [ ]:
# ===============================
# 4. Chargement des fichiers GTFS par ville
# ===============================

import os

def load_gtfs_city(city_folder):
    """
    Charge stops, trips, routes pour une ville GTFS
    """
    city_path = os.path.join(CLEAN_PATH, city_folder)

    stops = spark.read.option("header", True).csv(f"{city_path}/stops.txt")
    trips = spark.read.option("header", True).csv(f"{city_path}/trips.txt")
    routes = spark.read.option("header", True).csv(f"{city_path}/routes.txt")

    return stops, trips, routes


# Liste des villes disponibles
cities = [
    d for d in os.listdir(CLEAN_PATH)
    if os.path.isdir(os.path.join(CLEAN_PATH, d))
]

print("Villes détectées :", cities)


Villes détectées : ['Vectalia Movilidad (bus de la ville de Cáceres)', 'Xunta de Galicia Buses', 'Àrea Metropolitana de Barcelona (AMB)', 'Viagón Coaches', 'TUSSAM (Seville bus and tram)', 'TUS (Transportes Urbanos de Santander)', 'Transports Municipals del Gironés SAU (TMG) Girona city bus', 'Transports Municipaux D’Egara (TMESA) Terrassa bus urbain', 'TranspRober', 'TRAM Alicante', 'Tolosa City Council (Urbano de Tolosa)', 'Sopela Town Hall', 'TMESA', 'TIB Transports of the Balearic Islands - CIME Consell Insular de Menorca (Menorca Island Bus)', 'TIB Transports de les Illes Balears - CTM Transport Consortium of Mallorcam (Public transport on the island of Mallorca)', 'Réseau interurbain liO Occitanie', 'TIB Transports of the Balearic Islands - CIE Consell Insular d_Eivissa (Ibiza Island Bus)', 'Rafael Nadal Coaches', 'Rodil', 'Pinto City Council', 'Palma City Council (Palma de Mallorca city bus)', 'NVBW - Nahverkehrsgesellschaft Baden-Württemberg mbH', 'Oñati urbain (Oñatiko

In [ ]:
# ===============================
# Mode Jenkins (éviter timeout)
# ===============================
import os
JENKINS_MODE = os.environ.get("JENKINS_MODE", "0") == "1"
NUM_SAMPLE_CITIES = 5  # nombre de villes à traiter en mode Jenkins

if JENKINS_MODE:
    cities = cities[:NUM_SAMPLE_CITIES]
    print(f"⚡ Mode Jenkins activé : liste limitée à {len(cities)} villes : {cities}")

In [ ]:
# Exemple : première ville
city_name = cities[0]

stops_df, trips_df, routes_df = load_gtfs_city(city_name)

print("Ville :", city_name)
print("Stops :", stops_df.count())
print("Trips :", trips_df.count())
print("Routes :", routes_df.count())


Ville : Vectalia Movilidad (bus de la ville de Cáceres)
Stops : 237
Trips : 15260
Routes : 43


In [ ]:
stop_times.count()


31751871

In [ ]:
stop_times.show(10)


+-----------------+------------+--------------+-------+-------------+-------------+
|          trip_id|arrival_time|departure_time|stop_id|stop_sequence|source_folder|
+-----------------+------------+--------------+-------+-------------+-------------+
|2855_1_704_339306|     7:05:00|      07:05:00|      2|            0|            0|
|2855_1_704_339306|     7:08:00|      07:08:00|      3|            1|            0|
|2855_1_704_339306|     7:10:00|      07:10:00|      4|            2|            0|
|2855_1_704_339306|     7:12:00|      07:12:00|      5|            3|            0|
|2855_1_704_339306|     7:14:00|      07:14:00|      6|            4|            0|
|2855_1_704_339306|     7:15:00|      07:15:00|      7|            5|            0|
|2855_1_704_339306|     7:21:00|      07:21:00|      8|            6|            0|
|2855_1_704_339306|     7:22:00|      07:22:00|    240|            7|            0|
|2855_1_704_339306|     7:24:00|      07:24:00|    241|            8|       

In [ ]:
stop_times = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(STOP_TIMES_PATH)

print("✅ stop_times chargé (schéma inféré automatiquement)")
stop_times.printSchema()

# Vérification rapide
stop_times.show(5)

✅ stop_times chargé (schéma inféré automatiquement)
root
 |-- trip_id: string (nullable = true)
 |-- arrival_time: timestamp (nullable = true)
 |-- departure_time: timestamp (nullable = true)
 |-- stop_id: string (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- pickup_type: integer (nullable = true)
 |-- drop_off_type: integer (nullable = true)
 |-- source_folder: string (nullable = true)

+-----------------+-------------------+-------------------+-------+-------------+-----------+-------------+--------------------+
|          trip_id|       arrival_time|     departure_time|stop_id|stop_sequence|pickup_type|drop_off_type|       source_folder|
+-----------------+-------------------+-------------------+-------+-------------+-----------+-------------+--------------------+
|2855_1_704_339306|2025-12-16 07:05:00|2025-12-16 07:05:00|      2|            0|          0|            0|Vectalia Movilida...|
|2855_1_704_339306|2025-12-16 07:08:00|2025-12-16 07:08:00|      3|    

In [ ]:
# ===============================
# Jointures GTFS optimisées pour toutes les villes
# ===============================

from pyspark.sql.functions import col
import os

OUTPUT_BASE = f"{BASE_PATH}/NETWORK_BASE"
os.makedirs(OUTPUT_BASE, exist_ok=True)

# On boucle sur toutes les villes
for city in cities:
    print(f"\n🚀 Traitement de la ville : {city}")

    city_path = os.path.join(CLEAN_PATH, city)

    try:
        # Charger GTFS de la ville
        stops_df, trips_df, routes_df = load_gtfs_city(city)

        # Sélection minimale pour réduire mémoire
        stops_df = stops_df.select(
            "stop_id", "stop_name", "stop_lat", "stop_lon"
        )
        trips_df = trips_df.select(
            "trip_id", "route_id"
        )

        # Filtrer uniquement les lignes de stop_times correspondant à la ville
        # (source_folder = nom du dossier de la ville)
        stop_times_city = stop_times.filter(col("source_folder") == city)

        # Cache pour accélérer les jointures
        stop_times_city = stop_times_city.cache()

        # Jointure stop_times ↔ trips
        st_times_trips = stop_times_city.join(
            trips_df,
            on="trip_id",
            how="inner"
        )

        # Jointure avec stops
        network_base = st_times_trips.join(
            stops_df,
            on="stop_id",
            how="inner"
        )

        # Sauvegarde par ville (Parquet = rapide + compressé)
        output_path = os.path.join(OUTPUT_BASE, city)
        network_base.write.mode("overwrite").parquet(output_path)

        # Libération mémoire
        stop_times_city.unpersist()

        print(f"✅ Ville {city} traitée et sauvegardée")

    except Exception as e:
        print(f"❌ Erreur pour la ville {city} : {e}")



🚀 Traitement de la ville : Vectalia Movilidad (bus de la ville de Cáceres)
✅ Ville Vectalia Movilidad (bus de la ville de Cáceres) traitée et sauvegardée

🚀 Traitement de la ville : Xunta de Galicia Buses
✅ Ville Xunta de Galicia Buses traitée et sauvegardée

🚀 Traitement de la ville : Àrea Metropolitana de Barcelona (AMB)
✅ Ville Àrea Metropolitana de Barcelona (AMB) traitée et sauvegardée

🚀 Traitement de la ville : Viagón Coaches
✅ Ville Viagón Coaches traitée et sauvegardée

🚀 Traitement de la ville : TUSSAM (Seville bus and tram)
✅ Ville TUSSAM (Seville bus and tram) traitée et sauvegardée

🚀 Traitement de la ville : TUS (Transportes Urbanos de Santander)
✅ Ville TUS (Transportes Urbanos de Santander) traitée et sauvegardée

🚀 Traitement de la ville : Transports Municipals del Gironés SAU (TMG) Girona city bus
✅ Ville Transports Municipals del Gironés SAU (TMG) Girona city bus traitée et sauvegardée

🚀 Traitement de la ville : Transports Municipaux D’Egara (TMESA) Terrass

In [ ]:
OUTPUT_BASE = f"{BASE_PATH}/NETWORK_BASE"
city = cities[0]
df = spark.read.parquet(f"{OUTPUT_BASE}/{city}")

df.show(5, truncate=False)
df.printSchema()
print("Nombre de lignes :", df.count())


+-------+-----------------+-------------------+-------------------+-------------+-----------+-------------+------------------------------------------------+--------+------------------------------------+----------------+-----------------+
|stop_id|trip_id          |arrival_time       |departure_time     |stop_sequence|pickup_type|drop_off_type|source_folder                                   |route_id|stop_name                           |stop_lat        |stop_lon         |
+-------+-----------------+-------------------+-------------------+-------------+-----------+-------------+------------------------------------------------+--------+------------------------------------+----------------+-----------------+
|2      |2855_1_704_339306|2025-12-16 07:05:00|2025-12-16 07:05:00|0            |0          |0            |Vectalia Movilidad (bus de la ville de Cáceres)|10011   |Barrio Nuevo (Colegio Las Delicias) |39.4791475510164|-6.37496084186185|
|3      |2855_1_704_339306|2025-12-16 07:08:00|2

In [ ]:
# Vérifier si des lignes sont complètement vides (toutes les colonnes sont nulles)
empty_rows = df.filter(
    (df["stop_id"].isNull()) &
    (df["trip_id"].isNull()) &
    (df["arrival_time"].isNull()) &
    (df["departure_time"].isNull()) &
    (df["stop_sequence"].isNull()) &
    (df["pickup_type"].isNull()) &
    (df["drop_off_type"].isNull()) &
    (df["source_folder"].isNull()) &
    (df["route_id"].isNull()) &
    (df["stop_name"].isNull()) &
    (df["stop_lat"].isNull()) &
    (df["stop_lon"].isNull())
)

# Afficher le nombre de lignes vides
empty_row_count = empty_rows.count()
print(f"Nombre de lignes complètement vides : {empty_row_count}")

# Optionnel: Afficher quelques exemples de lignes vides
empty_rows.show(5, truncate=False)


Nombre de lignes complètement vides : 0
+-------+-------+------------+--------------+-------------+-----------+-------------+-------------+--------+---------+--------+--------+
|stop_id|trip_id|arrival_time|departure_time|stop_sequence|pickup_type|drop_off_type|source_folder|route_id|stop_name|stop_lat|stop_lon|
+-------+-------+------------+--------------+-------------+-----------+-------------+-------------+--------+---------+--------+--------+
+-------+-------+------------+--------------+-------------+-----------+-------------+-------------+--------+---------+--------+--------+



In [ ]:
import os
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, lead, unix_timestamp, lit
from pyspark.sql.window import Window

# ===============================
# Configuration des chemins
# ===============================
OUTPUT_BASE = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_BASE"  # Dossier contenant les données par ville
OUTPUT_EDGES_BASE = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES"  # Dossier de sortie pour les edges
os.makedirs(OUTPUT_EDGES_BASE, exist_ok=True)

# Liste des villes (dossiers)
cities = [f for f in os.listdir(OUTPUT_BASE) if os.path.isdir(os.path.join(OUTPUT_BASE, f))]
print("Villes détectées :", cities)

# ===============================
# Chargement de toutes les villes
# ===============================
dfs = []
for city in cities:
    city_df = spark.read.parquet(f"{OUTPUT_BASE}/{city}") \
        .withColumn("city", lit(city))  # Ajouter le nom de la ville
    dfs.append(city_df)
# ===============================
# Mode Jenkins (éviter timeout)
# ===============================
JENKINS_MODE = os.environ.get("JENKINS_MODE", "0") == "1"
NUM_SAMPLE_CITIES = 5  # nombre de villes à traiter en mode Jenkins

if JENKINS_MODE:
    cities = cities[:NUM_SAMPLE_CITIES]
    print(f"⚡ Mode Jenkins activé : liste limitée à {len(cities)} villes : {cities}")

# Combiner tous les DataFrames en un seul
df_all = reduce(DataFrame.unionByName, dfs)

# ===============================
# Création des edges (A → B)
# ===============================

# Sécurité : caster lat/lon en double
df_all = df_all.withColumn("stop_lat", col("stop_lat").cast("double")) \
               .withColumn("stop_lon", col("stop_lon").cast("double"))

# Fenêtre par trip (ordre des arrêts)
w = Window.partitionBy("trip_id").orderBy("stop_sequence")

# Récupérer les infos du stop suivant
edges = df_all \
    .withColumn("to_stop_id", lead("stop_id").over(w)) \
    .withColumn("to_lat", lead("stop_lat").over(w)) \
    .withColumn("to_lon", lead("stop_lon").over(w)) \
    .withColumn("to_arrival_time", lead("arrival_time").over(w)) \
    .withColumn("from_stop_id", col("stop_id")) \
    .withColumn("from_lat", col("stop_lat")) \
    .withColumn("from_lon", col("stop_lon")) \
    .withColumn("from_arrival_time", col("arrival_time"))

# Supprimer les dernières lignes de chaque trip (pas de successeur)
edges = edges.filter(col("to_stop_id").isNotNull())

# Calcul du temps de trajet (secondes)
edges = edges.withColumn(
    "travel_time_sec",
    unix_timestamp(col("to_arrival_time")) - unix_timestamp(col("from_arrival_time"))
)

# Calcul des deltas géographiques
edges = edges.withColumn("delta_lat", col("to_lat") - col("from_lat")) \
             .withColumn("delta_lon", col("to_lon") - col("from_lon"))

# Sélection finale (propre et légère)
edges_final = edges.select(
    "city",
    "trip_id",
    "route_id",
    "from_stop_id",
    "to_stop_id",
    "from_lat",
    "from_lon",
    "to_lat",
    "to_lon",
    "travel_time_sec",
    "delta_lat",
    "delta_lon"
)

print("✅ Edges créés pour toutes les villes")
edges_final.printSchema()
edges_final.show(5, truncate=False)
print("Nombre total d'edges :", edges_final.count())



Villes détectées : ['Vectalia Movilidad (bus de la ville de Cáceres)', 'Xunta de Galicia Buses', 'Àrea Metropolitana de Barcelona (AMB)', 'Viagón Coaches', 'TUSSAM (Seville bus and tram)', 'TUS (Transportes Urbanos de Santander)', 'Transports Municipals del Gironés SAU (TMG) Girona city bus', 'Transports Municipaux D’Egara (TMESA) Terrassa bus urbain', 'TranspRober', 'TRAM Alicante', 'Tolosa City Council (Urbano de Tolosa)', 'Sopela Town Hall', 'TMESA', 'TIB Transports of the Balearic Islands - CIME Consell Insular de Menorca (Menorca Island Bus)', 'TIB Transports de les Illes Balears - CTM Transport Consortium of Mallorcam (Public transport on the island of Mallorca)', 'Réseau interurbain liO Occitanie', 'TIB Transports of the Balearic Islands - CIE Consell Insular d_Eivissa (Ibiza Island Bus)', 'Rafael Nadal Coaches', 'Rodil', 'Pinto City Council', 'Palma City Council (Palma de Mallorca city bus)', 'NVBW - Nahverkehrsgesellschaft Baden-Württemberg mbH', 'Oñati urbain (Oñatiko

In [ ]:
# Nombre de villes distinctes dans edges_final
num_cities = edges_final.select("city").distinct().count()
print("Nombre de villes distinctes :", num_cities)

# Afficher la liste complète des villes
distinct_cities = edges_final.select("city").distinct().rdd.flatMap(lambda x: x).collect()
print("Liste des villes :", distinct_cities)


Nombre de villes distinctes : 47
Liste des villes : ['Junta de Extremadura (Bus du gouvernement régional d’Estrémadure)', 'Lurraldebus TBH (Interurban Tolosa Buruntzaldea)', 'Consorcio Regional de Transportes de Madrid CRTM Intercity Buses (Madrid Intercity Bus)', 'Avanza Grupo (Mataró city bus)', 'La Burundesa_SA', 'AUCORSA (Autobuses de Córdoba S.A.)', 'Lurraldebus Euskotren bus', 'Lurraldebus - Hernani Urban (bus urbain d’Hernani)', 'TIB Transports de les Illes Balears - CTM Transport Consortium of Mallorcam (Public transport on the island of Mallorca)', 'Autocorb Coaches', 'FGV - Generalitat Valenciana Trains and trams in Alicante and the Costa Blanca', 'Cots Alsina', 'Lurraldebus Zarautz', 'TIB Transports of the Balearic Islands - CIME Consell Insular de Menorca (Menorca Island Bus)', 'Métro Bilbao', 'TRAM Alicante', 'Sopela Town Hall', 'Lurraldebus Ekialdebus', 'La Coruña Tram Company SA', 'Ferry Fred. Olsen (Fred Olsen Express)', 'Oñati urbain (Oñatiko herribusa)', 'AISA

In [ ]:
# Nombre de villes distinctes
num_cities = edges_final.select("city").distinct().count()
print("Nombre de villes distinctes :", num_cities)


Nombre de villes distinctes : 47


In [ ]:
import os
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, lead, unix_timestamp, lit
from pyspark.sql.window import Window

# ===============================
# Configuration des chemins
# ===============================
OUTPUT_BASE = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_BASE"  # Dossier contenant les données par ville
OUTPUT_EDGES_BASE = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES"  # Dossier de sortie pour les edges
os.makedirs(OUTPUT_EDGES_BASE, exist_ok=True)

# ===============================
# Liste des villes (dossiers)
# ===============================
cities = [f for f in os.listdir(OUTPUT_BASE) if os.path.isdir(os.path.join(OUTPUT_BASE, f))]
print("Nombre de villes détectées :", len(cities))

# Vérifier que le nombre de villes est supérieur à 100
if len(cities) < 100:
    raise ValueError(f"⚠️ Seulement {len(cities)} villes détectées ! Veuillez vérifier vos données avant de créer le réseau.")

print("✅ Plus de 100 villes détectées, on peut créer le réseau.")

# ===============================
# Chargement de toutes les villes
# ===============================
dfs = []
for city in cities:
    city_df = spark.read.parquet(f"{OUTPUT_BASE}/{city}") \
        .withColumn("city", lit(city))  # Ajouter le nom de la ville
    dfs.append(city_df)

# Combiner tous les DataFrames en un seul
df_all = reduce(DataFrame.unionByName, dfs)

# ===============================
# Création des edges (A → B)
# ===============================

# Sécurité : caster lat/lon en double
df_all = df_all.withColumn("stop_lat", col("stop_lat").cast("double")) \
               .withColumn("stop_lon", col("stop_lon").cast("double"))

# Fenêtre par trip (ordre des arrêts)
w = Window.partitionBy("trip_id").orderBy("stop_sequence")

# Récupérer les infos du stop suivant
edges = df_all \
    .withColumn("to_stop_id", lead("stop_id").over(w)) \
    .withColumn("to_lat", lead("stop_lat").over(w)) \
    .withColumn("to_lon", lead("stop_lon").over(w)) \
    .withColumn("to_arrival_time", lead("arrival_time").over(w)) \
    .withColumn("from_stop_id", col("stop_id")) \
    .withColumn("from_lat", col("stop_lat")) \
    .withColumn("from_lon", col("stop_lon")) \
    .withColumn("from_arrival_time", col("arrival_time"))

# Supprimer les dernières lignes de chaque trip (pas de successeur)
edges = edges.filter(col("to_stop_id").isNotNull())

# Calcul du temps de trajet (secondes)
edges = edges.withColumn(
    "travel_time_sec",
    unix_timestamp(col("to_arrival_time")) - unix_timestamp(col("from_arrival_time"))
)

# Calcul des deltas géographiques
edges = edges.withColumn("delta_lat", col("to_lat") - col("from_lat")) \
             .withColumn("delta_lon", col("to_lon") - col("from_lon"))

# Sélection finale (propre et légère)
edges_final = edges.select(
    "city",
    "trip_id",
    "route_id",
    "from_stop_id",
    "to_stop_id",
    "from_lat",
    "from_lon",
    "to_lat",
    "to_lon",
    "travel_time_sec",
    "delta_lat",
    "delta_lon"
)

# ===============================
# Vérification du résultat
# ===============================
print("✅ Edges créés pour toutes les villes")
edges_final.printSchema()
edges_final.show(5, truncate=False)
print("Nombre total d'edges :", edges_final.count())

# Nombre exact de villes dans edges_final
num_cities = edges_final.select("city").distinct().count()
print("Nombre exact de villes dans edges_final :", num_cities)


Nombre de villes détectées : 105
✅ Plus de 100 villes détectées, on peut créer le réseau.
✅ Edges créés pour toutes les villes
root
 |-- city: string (nullable = false)
 |-- trip_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- from_stop_id: string (nullable = true)
 |-- to_stop_id: string (nullable = true)
 |-- from_lat: double (nullable = true)
 |-- from_lon: double (nullable = true)
 |-- to_lat: double (nullable = true)
 |-- to_lon: double (nullable = true)
 |-- travel_time_sec: long (nullable = true)
 |-- delta_lat: double (nullable = true)
 |-- delta_lon: double (nullable = true)

+---------------------------------------+------------------------------------+------------------------------------+------------------------------------+------------------------------------+----------+----------+----------+----------+---------------+---------------------+----------------------+
|city                                   |trip_id                             |route_id 

In [ ]:
import os

# Dossier de sortie pour tous les edges (déjà créé)
os.makedirs(OUTPUT_EDGES_BASE, exist_ok=True)

# Récupérer la liste des villes présentes dans edges_final
cities_in_df = edges_final.select("city").distinct().rdd.flatMap(lambda x: x).collect()
print("Nombre de villes à sauvegarder :", len(cities_in_df))

# Sauvegarde par ville
for city_name in cities_in_df:
    # Créer un nom de dossier sûr (remplacer espaces et caractères spéciaux)
    safe_city_name = city_name.replace("/", "_").replace(" ", "_").replace("'", "_")

    # Filtrer les edges pour cette ville
    city_edges = edges_final.filter(edges_final.city == city_name)

    # Chemin de sortie pour cette ville
    output_path = os.path.join(OUTPUT_EDGES_BASE, safe_city_name)

    # Sauvegarde en Parquet
    city_edges.write.mode("overwrite").parquet(output_path)

    # Afficher le nombre de lignes sauvegardées
    num_rows = city_edges.count()
    print(f"✅ Edges de {city_name} sauvegardés dans {output_path} ({num_rows} lignes)")

print("🎉 Toutes les villes ont été sauvegardées !")


Nombre de villes à sauvegarder : 47
✅ Edges de Junta de Extremadura (Bus du gouvernement régional d’Estrémadure) sauvegardés dans /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES/Junta_de_Extremadura_(Bus_du_gouvernement_régional_d’Estrémadure) (3975 lignes)
✅ Edges de Lurraldebus TBH (Interurban Tolosa Buruntzaldea) sauvegardés dans /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES/Lurraldebus_TBH_(Interurban_Tolosa_Buruntzaldea) (100331 lignes)
✅ Edges de Consorcio Regional de Transportes de Madrid CRTM Intercity Buses (Madrid Intercity Bus) sauvegardés dans /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES/Consorcio_Regional_de_Transportes_de_Madrid_CRTM_Intercity_Buses_(Madrid_Intercity_Bus) (606935 lignes)
✅ Edges de Avanza Grupo (Mataró city bus) sauvegardés dans /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES/Avanza_Grupo_(Mataró_city_bus) (31611 lignes)
✅ Edges de La Burundesa_SA sauvegardés dans /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES/La_Burundesa_SA (1329 lignes)
✅ E